## **Detección automática de anomalías visuales en pavimento mediante análisis cromático**

Este script implementa un algoritmo para **resaltar posibles anomalías** (manchas, fisuras, acumulaciones, decoloración) sobre el **pavimento** en imágenes satelitales `.tif`, previamente recortadas a las vías. Utiliza análisis cromático en el espacio de color HSV para identificar tonalidades inusuales que podrían estar asociadas a condiciones anormales sobre la vía.

> 🛰️ **Importante**: El análisis se aplica sobre imágenes que **ya contienen solo pavimento**, segmentado en etapas anteriores con LangSAM. Esto asegura que el análisis se concentre exclusivamente en la calzada.

### 🎯 **Objetivo del proceso**
Detectar automáticamente **diferencias cromáticas** que puedan indicar **potenciales fallas** o alteraciones en el pavimento, y generar salidas visuales y cuantitativas que puedan alimentar análisis posteriores o validaciones en campo.

### ✅ **Funcionalidades principales**
- 🎨 Conversión de la imagen al espacio HSV para una segmentación más robusta por color.
- 🚫 Aplicación de filtros para eliminar colores comunes del fondo (negros, blancos, grises, verdes, rojos, azules).
- 🖼️ Exportación de:
  - Imagen filtrada `.tif` resaltando las zonas inusuales.
  - Comparación visual `.jpg` entre imagen original y filtrada.
  - Archivo `.txt` con conteo y clasificación de colores remanentes.

### 🛠️ **Variables clave**
- `ruta_entrada`: imagen de entrada `.tif`.
- `ruta_imagen_salida`: imagen procesada con anomalías resaltadas.
- `ruta_txt_salida`: tabla con descripción y porcentaje de cada color.
- `ruta_comparacion_jpg`: imagen comparativa (opcional).

### 🧪 **Aplicaciones típicas**
- Identificación preliminar de posibles baches, manchas, fisuras o reparaciones.
- Entrenamiento o validación de modelos de clasificación de fallas.
- Análisis cromático del deterioro de corredores viales.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from collections import Counter
from matplotlib.colors import rgb_to_hsv
from tqdm import tqdm
from shapely.geometry import Polygon
import geopandas as gpd

def descripcion_personalizada(color_rgb):
    r, g, b = color_rgb

    # --- NEGROS, BLANCOS Y GRISES ---
    # Negros profundos y negros más claros
    if r < 30 and g < 30 and b < 30:
        return "Negro profundo"
    if r < 60 and g < 60 and b < 60:
        return "Negro"
    # Blancos
    if r > 240 and g > 240 and b > 240:
        return "Blanco puro"
    if r > 220 and g > 220 and b > 220:
        return "Blanco"
    # Gamas de grises (los tres canales muy similares)
    if abs(r - g) < 10 and abs(g - b) < 10:
        if r < 50:
            return "Gris carbón"
        elif r < 100:
            return "Gris oscuro"
        elif r < 180:
            return "Gris medio"
        else:
            return "Gris claro"

    # --- VERDES ---
    if g > r and g > b:
        # Verdes muy brillantes
        if g > 220 and r < 50 and b < 50:
            return "Verde neón"
        if g > 200 and r < 100:
            return "Verde pasto brillante"
        # Verdes con matices amarillentos
        if g > 170 and r > 100 and r < 170:
            return "Verde musgo claro"
        if g > 150 and r > 100:
            return "Verde oliva claro"
        if g > 120 and b < 70 and r < 100:
            return "Verde lima"
        # Verdes intermedios
        if g > 100 and r > 80:
            return "Verde oliva"
        if g > 80:
            return "Verde militar"
        # Verdes oscuros
        return "Verde oscuro"

    # --- ROJOS / ROSADOS / MORADOS ---
    # Primero controlamos rojos/rosados brillantes o intensos
    if r > 200 and g < 100 and b < 100:
        return "Rojo intenso"
    if r > 180 and g < 120 and b < 120:
        return "Rojo ladrillo"
    if r > 160 and g < 100 and b < 100:
        return "Rojo vino"
    # Rosas y fucsias
    if r > 200 and b > 150 and g < 180:
        return "Rosa fuerte"
    if r > 180 and b > 150:
        return "Rosa pastel"
    # Tonos rojizos intermedios
    if r > 180 and g > 80 and g < 140 and b < 100:
        return "Rojo teja"
    if r > 160 and g > 100 and b < 80:
        return "Rojo anaranjado"
    if r > 120 and g < 70 and b < 70:
        return "Rojo oscuro quemado"

    # Violeta / Morado
    if r > b and b > g:  # r > b > g puede quedarse como una aproximación a violetas
        if r > 200 and b > 160:
            return "Rosado fuerte"  # ya contemplado arriba, pero lo dejamos como fallback
        elif r > 160 and b > 120:
            return "Lila claro"
        else:
            return "Violeta oscuro"

    # --- NARANJAS Y OCRES ---
    if r > 200 and g > 120 and g < 180 and b < 100:
        return "Naranja dorado"
    if r > 180 and g > 100 and g < 150 and b < 80:
        return "Ocre tierra"
    if r > 180 and g > 80 and g < 120 and b < 60:
        return "Naranja ladrillo"
    if r > 200 and g > 160 and b < 100:
        return "Naranja claro"
    if r > 180 and g > 130 and b < 90:
        return "Naranja amarillento"

    # --- AMARILLOS Y BEIGE ---
    # Amarillos muy claros
    if r > 240 and g > 230 and b < 150:
        return "Amarillo pálido"
    if r > 220 and g > 200 and b < 100:
        return "Amarillo claro"
    if r > 200 and g > 200 and b < 100:
        return "Amarillo vibrante"
    if r > 180 and g > 160 and b < 80:
        return "Amarillo mostaza"
    if r > 150 and g > 150 and b < 100:
        return "Amarillo tierra"
    # Beige
    if r > 200 and g > 180 and b > 140:
        return "Beige claro"
    if r > 180 and g > 150 and b > 100:
        return "Beige dorado"

    # --- AZULES / CIANES ---
    if b > r and b > g:
        # Azules muy claros y/o pastel
        if b > 220 and g > 200 and r < 100:
            return "Azul cielo muy claro"
        if b > 200 and g > 180 and r < 100:
            return "Azul cielo claro"
        # Tonos turquesa / cian
        if b > 180 and g > 150 and r < 80:
            return "Azul turquesa"
        if b > 160 and g > 160 and r < 60:
            return "Cian"
        # Escalas de azul
        if b > 180:
            return "Azul claro"
        if b > 130:
            return "Azul medio"
        if b > 90 and g > 60:
            return "Azul profundo"
        if b > 80:
            return "Azul marino"
        return "Azul oscuro"

    # --- MARRONES / CAFÉS ---
    if r > 100 and g > 70 and b < 60:
        # Más específico primero
        if r > 160 and g > 120 and b < 80:
            return "Marrón anaranjado"
        if r > 140 and g > 100 and b < 80:
            return "Marrón claro"
        if r > 80 and g > 50 and b < 30:
            return "Marrón oscuro"
        return "Marrón"

    # Si no coincide con ninguno de los anteriores
    return "Color indefinido"

def filtrar_colores(imagen_rgb):
    img = imagen_rgb / 255.0
    hsv = rgb_to_hsv(img)

    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    intensidad = img.mean(axis=2)

    mascara_negro = intensidad < 0.15
    mascara_blanco = intensidad > 0.85
    mascara_gris = (intensidad > 0.15) & (intensidad < 0.85) & (s < 0.25)

    mascara_verde = (h > 0.10) & (h < 0.52) & (s > 0.15) & (v > 0.1)
    mascara_rojo = ((h < 0.05) | (h > 0.95)) & (s > 0.15) & (v > 0.1)
    mascara_azul = (h > 0.55) & (h < 0.75) & (s > 0.15) & (v > 0.1)

    mascara_total = (
        mascara_negro |
        mascara_blanco |
        mascara_gris |
        mascara_verde |
        mascara_rojo |
        mascara_azul
    )

    filtrada = img.copy()
    filtrada[mascara_total] = 0

    return (filtrada * 255).astype(np.uint8), ~mascara_total


def procesar_imagen_color_filtro(
    ruta_entrada,
    ruta_imagen_salida,
    ruta_txt_salida,
    ruta_shp_salida,
    ruta_comparacion_jpg=None,
    comparacion=False
):
    with rasterio.open(ruta_entrada) as src:
        img = src.read([1, 2, 3]).transpose(1, 2, 0)
        perfil = src.profile.copy()

    img_filtrada_uint8, _ = filtrar_colores(img)

    # Guardar imagen filtrada como GeoTIFF
    perfil.update({
        "count": 3,
        "dtype": 'uint8',
        "driver": "GTiff"
    })

    with rasterio.open(ruta_imagen_salida, 'w', **perfil) as dst:
        dst.write(img_filtrada_uint8.transpose(2, 0, 1))

    # Comparación original vs filtrada
    if comparacion and ruta_comparacion_jpg:
        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        axs[0].imshow(img.astype(np.uint8))
        axs[0].set_title("Original")
        axs[0].axis('off')
        axs[1].imshow(img_filtrada_uint8)
        axs[1].set_title("Filtrada")
        axs[1].axis('off')
        plt.tight_layout()
        plt.savefig(ruta_comparacion_jpg)
        plt.close()

    # Conteo de colores
    img_pixels = img_filtrada_uint8.reshape(-1, 3)
    colores_filtrados = [tuple(p) for p in img_pixels if not np.all(p == 0)]
    conteo = Counter(colores_filtrados)

    total_pixeles_imagen = img.shape[0] * img.shape[1]
    total_pixeles_filtrada = len(colores_filtrados)

    columnas = {'imagen': [], 'color_rgb': [], 'descripcion': [], 'pixeles': [], 'pct_total': [], 'pct_filtrada': [], 'geometry': []}

    # Guardar archivo TXT con porcentajes
    with open(ruta_txt_salida, 'w') as f:
        f.write("imagen\tcolor_rgb\tdescripcion\tpixeles\tporcentaje_total_imagen\tporcentaje_filtrada\n")
        for color, cantidad in conteo.items():
            desc = descripcion_personalizada(color)
            pct_total = round((cantidad / total_pixeles_imagen) * 100, 4)
            pct_filtrada = round((cantidad / total_pixeles_filtrada) * 100, 4) if total_pixeles_filtrada > 0 else 0

            f.write(f"{os.path.basename(ruta_entrada)}\t{color}\t{desc}\t{cantidad}\t{pct_total}\t{pct_filtrada}\n")

            # Añadir la información al shapefile
            columnas['imagen'].append(os.path.basename(ruta_entrada))
            columnas['color_rgb'].append(str(color))
            columnas['descripcion'].append(desc)
            columnas['pixeles'].append(cantidad)
            columnas['pct_total'].append(pct_total)
            columnas['pct_filtrada'].append(pct_filtrada)
            columnas['geometry'].append(Polygon([(0, 0), (0, 1), (1, 1), (1, 0)]))  # Este es un ejemplo

    # Crear el GeoDataFrame
    gdf = gpd.GeoDataFrame(columnas, geometry='geometry', crs="EPSG:4326")

    # Guardar el shapefile
    gdf.to_file(ruta_shp_salida, driver='ESRI Shapefile')

    print(f"✅ Procesada: {os.path.basename(ruta_entrada)}")



# === BLOQUE PRINCIPAL ===
if __name__ == "__main__":
    # Define la ruta con el pavimento obtenido
    root = "./data/IMAGES/ONLY_ROADS_SAMpavement_terrz19_Z21_BB22"
    troncal = "Troncal1"
    list_img = glob.glob(os.path.join(root, troncal, "*.tif"))
    
    print(f"🔍 {len(list_img)} imágenes encontradas en {os.path.join(root, troncal)}")

    fol_seg = os.path.join(root, "seg_col", troncal)
    
    
    os.makedirs(fol_seg, exist_ok=True)

    for ruta in tqdm(list_img, desc="Procesando imágenes"):
        try:
            nombre_base = os.path.splitext(os.path.basename(ruta))[0]
            ruta_imagen_salida = os.path.join(fol_seg, f"{nombre_base}.tif")
            
            # Verificar si la imagen ya fue procesada
            if os.path.exists(ruta_imagen_salida):
                print(f"⚠️ {ruta_imagen_salida} ya existe. Saltando...")
                continue

            procesar_imagen_color_filtro(
                ruta_entrada=ruta,
                ruta_imagen_salida=ruta_imagen_salida,
                ruta_txt_salida=ruta_imagen_salida.replace(".tif", "_colores.txt"),
                ruta_comparacion_jpg=ruta_imagen_salida.replace(".tif", "_comp.jpg"),
                ruta_shp_salida=ruta_imagen_salida.replace(".tif", ".shp"),
                comparacion=True
            )
        except:
            pass

# Filtros y consolidación

In [ ]:
import pandas as pd
import glob
import os

ruta_archivos='./data/IMAGES/ONLY_ROADS_SAMpavement_terrz19_Z21_BB22/seg_col/Troncal1'

# Buscar todos los archivos *_colores.txt
archivos_txt = glob.glob(os.path.join(ruta_archivos, "*_colores.txt"))

# Cargar todos los archivos en un solo DataFrame
dfs = []
for archivo in archivos_txt:
    try:
        df = pd.read_csv(archivo, sep="\t")
        dfs.append(df)
    except Exception as e:
        print(f"❌ Error cargando {archivo}: {e}")

# Combinar todos
df_completo = pd.concat(dfs, ignore_index=True)

# Agrupar por imagen y descripción
df_agrupado = df_completo.groupby(
    ["imagen", "descripcion"], as_index=False
).agg({
    "pixeles": "sum",
    "porcentaje_total_imagen": "sum",
    "porcentaje_filtrada": "sum"
})

out_excel=ruta_archivos+'.xlsx'
df_agrupado.to_excel(out_excel)

# Agrupar por imagen
df_agrupado2 = df_agrupado.groupby(
    ["imagen"], as_index=False
).agg({
    "pixeles": "sum",
    "porcentaje_total_imagen": "sum",
    "porcentaje_filtrada": "sum"
})

# Guardar el resultado agrupado
out_csv=ruta_archivos+'_colores_agrupados.csv'
df_agrupado2.to_csv(out_csv, index=False, sep=",")

df_agrupado2

In [ ]:
import pandas as pd
import os, glob
from pathlib import Path

# Ruta de la carpeta donde están los archivos Excel
carpeta = "./data/IMAGES/ONLY_ROADS_SAMpavement_terrz19_Z21_BB22/seg_col"  # Reemplaza con la ruta real
archivos_excel = glob.glob(os.path.join(carpeta,'*.xlsx'))[:6]

# Lista para guardar los DataFrames temporales
lista_dataframes = []

for archivo in archivos_excel:
    try:
        df = pd.read_excel(archivo)
        df['archivo_origen'] = os.path.basename(archivo)  # Opcional: para saber de qué archivo viene cada fila
        lista_dataframes.append(df)
    except Exception as e:
        print(f"Error leyendo {os.path.basename(archivo)}: {e}")

# Unimos todos los DataFrames en uno solo
df_unido = pd.concat(lista_dataframes, ignore_index=True)
df_unido['id'] = df_unido['imagen'].str.split('.', n=1).str[0]


# Mostrar el resultado
print(f"Total de archivos procesados: {len(archivos_excel)}")
print(f"Total de filas unidas: {df_unido.shape[0]}")

excel_full=os.path.join(carpeta,'Troncales_full_segcol.xlsx')
df_unido.to_excel(excel_full)
df_unido

In [ ]:
#Excluir
#filtro = ~df_unido['descripcion'].str.contains(r'^(Negro|Azul|Gris|violeta)', case=False, regex=True)
filtro = ~df_unido['descripcion'].str.contains(r'^(Color|Negro|Azul|Gris|violeta|Rojo|Lila|Rosa)', case=False, regex=True)

df_filtrado = df_unido[filtro]

# Agrupar por 'imagen', 'descripcion', 'id' y obtener la suma de 'pixeles' y 'porcentaje_total_imagen'
df_agrupado = df_filtrado.groupby(['imagen', 'descripcion', 'id'], as_index=False).agg({
    'pixeles': 'sum',
    'porcentaje_total_imagen': 'sum'
})

# Agrupar por 'imagen', 'id' y obtener la suma de 'pixeles' y 'porcentaje_total_imagen'
df_agrupado = df_filtrado.groupby(['imagen', 'id'], as_index=False).agg({
    'pixeles': 'sum',
    'porcentaje_total_imagen': 'sum'
})
# Mostrar el DataFrame agrupado
df_agrupado


# Filtrar df_agrupado por 'porcentaje_total_imagen'
df_filtrado_porcentaje = df_agrupado[(df_agrupado['porcentaje_total_imagen'] > 0.3) & (df_agrupado['porcentaje_total_imagen'] < 0.5)]
df_filtrado_porcentaje.rename(columns={'porcentaje_total_imagen': 'anomalies_pct'}, inplace=True)
print(len(df_filtrado_porcentaje.id.value_counts()))

anomalies_excel='./data/Troncales_anomalies.xlsx'
df_filtrado_porcentaje.to_excel(anomalies_excel)

df_filtrado_porcentaje